In [0]:
%spark.pyspark
from pyspark.sql import functions as F
from pyspark.sql import Window


OUT_PATH = "hdfs:///tmp/sandbox_zeppelin/mart_city_top_products/"

TOP_N = 2
users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)
orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)
products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)
users.show()
orders.show()
products.show()

In [1]:
%spark.pyspark

orders_enriched = orders.withColumn(
    "revenue",
    F.col("qty").cast("double") * F.col("price").cast("double")
)


base = (
    orders_enriched
    .join(users, on="user_id", how="inner")
    .join(products, on="product_id", how="inner")
)


agg = (
    base
    .groupBy("city", "product_id", "product_name")
    .agg(
        F.countDistinct("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum"),
    )
)


w = Window.partitionBy("city").orderBy(F.col("revenue_sum").desc(), F.col("product_id").asc())

mart_city_top_products = (
    agg
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= F.lit(TOP_N))
    .drop("rn")
)


mart_city_top_products = mart_city_top_products.select(
    "city", "product_id", "product_name", "orders_cnt", "qty_sum", "revenue_sum"
)

mart_city_top_products.show(truncate=False)

In [2]:
%spark.pyspark
(
    mart_city_top_products
    .write
    .mode("overwrite")
    .parquet(OUT_PATH)
)
print(f"Wrote parquet to: {OUT_PATH}")

In [3]:
%spark.pyspark
mart_read = spark.read.parquet(OUT_PATH)


top1 = mart_read.orderBy(F.col("revenue_sum").desc(), F.col("city").asc()).limit(1)

mart_read.show(truncate=False)
top1.show(truncate=False)

In [4]:
%spark.pyspark
